# Component Landscape

Interactive exploration of the speaker landscape by ICA component.

In [1]:
# === Colab setup — automatically skipped when running locally ===
# Reads the project and its data from the shared PONS folder in Drive. 
# Require a shortcut to PONS folder from myDrive
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    PROJECT_DIR = ("/content/drive/MyDrive/PONS/EXPERIMENTS/TERM-CORRELATION"
                   "/Interactive_Component_Landscape")

    # 1. Mount Google Drive.
    from google.colab import drive
    drive.mount("/content/drive")

    assert os.path.isdir(PROJECT_DIR), (
        f"{PROJECT_DIR} not found — add a shortcut to the shared PONS folder "
        "in My Drive (Shared with me > PONS > Organise > Add shortcut).")

    # 2. Read the data from that folder, and make the src/ modules importable.
    os.environ["DATA_DIR"] = f"{PROJECT_DIR}/data"
    sys.path.insert(0, f"{PROJECT_DIR}/src")

    # 3. Install the packages Colab doesn't already ship with.
    %pip install -q -r "{PROJECT_DIR}/requirements-colab.txt"

    # 4. ipywidgets interactivity on Colab needs the custom widget manager.
    from google.colab import output
    output.enable_custom_widget_manager()

In [2]:
# Setup
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets as widgets

from ica.landscape import build_landscape, configure_fonts

configure_fonts()

In [3]:
# Data: embedding, ICA, retained components, 2D projection, stance labels
landscape = build_landscape()
print(f"{len(landscape.selected_components)} retained components: {landscape.selected_components}")

13 retained components: [6, 19, 26, 29, 46, 47, 69, 78, 81, 100, 102, 166, 201]


In [ ]:
# Render: the shared Landscape.render, displayed inline
def render(min_similarity, **params):
    # 0 on the similarity slider = no floor, as the webapp's unticked checkbox
    fig, note, tables = landscape.render(min_similarity=min_similarity or None, **params)
    plt.show()
    plt.close(fig)

    if note:
        print(note)
    for table in tables:
        print(table["heading"])
        if table["nearest"] is not None:
            display(table["nearest"])

In [ ]:
# Widgets and wiring. The defaults are the webapp landscape page's
# (src/webapp/params.py DEFAULTS), so notebook and app open on the same picture.
w_component = widgets.Dropdown(options=landscape.selected_components,
                               value=201 if 201 in landscape.selected_components
                               else landscape.selected_components[0],
                               description="Component")
w_phase = widgets.Dropdown(options=["1", "2", "3", "4", "all", "pooled"], value="pooled",
                           description="Phase")
w_strong_speakers = widgets.Checkbox(value=False, description="Strong speakers (orange)")
w_strong_labels = widgets.Checkbox(value=False, description="Strong word (dark orange)")
w_extreme_speakers = widgets.Checkbox(value=False, description="Extreme speakers (pink)")
w_n_extreme_speakers = widgets.BoundedIntText(value=50, min=1, max=5000, description="N speakers", disabled=True)
w_extreme_speakers.observe(lambda ch: setattr(w_n_extreme_speakers, "disabled", not ch["new"]), names="value")
w_extreme_labels = widgets.Checkbox(value=False, description="Extreme word (pink)")
w_n_extreme_words = widgets.BoundedIntText(value=10, min=1, max=100, description="N words", disabled=True)
w_extreme_labels.observe(lambda ch: setattr(w_n_extreme_words, "disabled", not ch["new"]), names="value")
# Label readability: 1.0 = adjustText default, higher = more separated, 0 = off
w_spread = widgets.FloatSlider(value=0.25, min=0.0, max=4.0, step=0.25,
                               description="Label spread", continuous_update=False)

w_unit_norm = widgets.Checkbox(value=True, description="Unit norm")
w_mean_centre = widgets.Checkbox(value=True, description="Mean centring")
w_centroid_source = widgets.RadioButtons(options=["strong speakers", "extreme speakers"],
                                         value="extreme speakers", description="Average of")
w_n_extreme = widgets.BoundedIntText(value=50, min=1, max=5000, description="N extreme")
w_centroid_source.observe(lambda ch: setattr(w_n_extreme, "disabled", ch["new"] == "strong speakers"), names="value")

# These labels overflow the default ipywidgets label width
_wide_label = {"description_width": "130px"}
# (label, value) pairs: render receives the short value, the UI shows the label
w_word_filter = widgets.RadioButtons(options=[("strong words", "strong words"),
                                              ("extreme words", "extreme words"),
                                              ("all words", "all words"),
                                              ("off", "off")],
                                     value="extreme words", description="Search among", style=_wide_label)
w_k_extreme = widgets.BoundedIntText(value=200, min=1, max=10000, description="K extreme", style=_wide_label)
w_word_filter.observe(lambda ch: setattr(w_k_extreme, "disabled", ch["new"] != "extreme words"), names="value")
w_min_count = widgets.BoundedIntText(value=10, min=1, max=100000, description="Min. word frequency",
                                     continuous_update=False, style=_wide_label)
# Centroid-similarity floor on the found words; 0 turns it off
w_min_similarity = widgets.FloatSlider(value=0.5, min=0.0, max=0.9, step=0.05,
                                       description="Min. similarity", readout_format=".2f",
                                       continuous_update=False, style=_wide_label)
w_topn = widgets.BoundedIntText(value=50, min=1, max=200, description="Top n", style=_wide_label)

_column_layout = widgets.Layout(margin="0 50px 10px 0")
ui = widgets.HBox([
    widgets.VBox([widgets.HTML("<b>Display</b>"), w_component, w_phase,
                  w_strong_speakers, w_strong_labels,
                  w_extreme_speakers, w_n_extreme_speakers,
                  w_extreme_labels, w_n_extreme_words, w_spread],
                 layout=_column_layout),
    widgets.VBox([widgets.HTML("<b>Centroid</b>"), w_unit_norm, w_mean_centre,
                  w_centroid_source, w_n_extreme], layout=_column_layout),
    widgets.VBox([widgets.HTML("<b>Nearest words</b>"), w_word_filter, w_k_extreme,
                  w_min_count, w_min_similarity, w_topn], layout=_column_layout),
], layout=widgets.Layout(flex_flow="row wrap"))

out = widgets.interactive_output(render, {
    "component": w_component, "phase": w_phase,
    "show_strong_speakers": w_strong_speakers, "strong_labels": w_strong_labels,
    "show_extreme_speakers": w_extreme_speakers, "extreme_labels": w_extreme_labels,
    "n_extreme_speakers": w_n_extreme_speakers, "n_extreme_words": w_n_extreme_words,
    "unit_norm": w_unit_norm, "mean_centre": w_mean_centre,
    "centroid_source": w_centroid_source, "n_extreme": w_n_extreme,
    "word_filter": w_word_filter, "k_extreme": w_k_extreme,
    "min_count": w_min_count, "min_similarity": w_min_similarity,
    "topn": w_topn, "spread": w_spread,
})
display(ui, out)